<a href="https://colab.research.google.com/github/jasminealyaa1212/DIP-final-project/blob/main/DIP%20final%20project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# %% CELL 1 - Config
import glob
import os
import time
import urllib.request
from collections import deque

import cv2
import numpy as np

try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
except ImportError:
    pass

BASE = '/content/drive/MyDrive/semester 3/Digital Image Processing/final project'
VIDEO_PATH = f'{BASE}/video 6.mp4'

# Allowed traffic direction as a vector (dx, dy); image y grows DOWNWARD.
# (0, 1) = down the screen, (0, -1) = up, (-2, -1) = toward upper-left, etc.
ALLOWED_VECTOR = (0, 1)

# Optional per-lane directions (coordinates in the W x H processing frame below).
# LANES = [{'name': 'L', 'vector': (0, 1),  'polygon': [(50, 350), (250, 350), (275, 100), (150, 100)]},
#          {'name': 'R', 'vector': (0, -1), 'polygon': [(260, 350), (500, 350), (400, 100), (285, 100)]}]
LANES = []

W, H = 640, 360              # processing size (raise to 960x540 if vehicles are tiny)
TARGET_FPS = 10              # analysed frames per second of video (skip is derived from the source fps)
WARMUP = 15                  # processed frames used to learn the background
MIN_BLOB_AREA = 700          # smallest moving blob (px^2) treated as a vehicle
USE_BLOB_FALLBACK = True     # keep moving blobs even if no cascade recognises them
HIST_LEN, MIN_HIST, MAX_MISSED = 10, 5, 8
# Things that are NOT traffic but change over time (screen-recorder popups, timestamp, logo...).
# Rectangles (x1, y1, x2, y2) in the W x H processing frame are blanked out of the motion mask.
# Defaults match the Simpang Terban clip; use [] to disable, and re-measure for your own clips.
IGNORE_RECTS = [(0, 0, 335, 26), (405, 0, 470, 42), (538, 33, 640, 68), (612, 330, 640, 360)]
MAX_FG_FRACTION = 0.25       # skip frames where >25% of pixels "move" (exposure / colour flicker); 1.0 = off
EDGE_MARGIN = 3             # boxes touching the frame border are partly out of view: don't judge them
MAX_AREA_JUMP = 1.8          # blob area changing by this factor = merge/split: restart the trajectory
MIN_AGE_TO_COUNT = 8         # a track must be matched this many times before it counts as a vehicle
MIN_PATH_TO_COUNT = 40       # ...and must have travelled this many px (ignores flickers / static blobs)
MIN_TRAVEL_RATIO = 0.2       # net travel must exceed this * box height
COS_THR = 0.5                # direction must be within 60 deg of the road axis
VOTES_TO_FLAG = 3            # consecutive wrong-way verdicts -> violation (1 = voting filter OFF)

# ---- Filtering switches: flip ONE at a time to run the with/without comparison ----
FILTER_BLUR = True           # Gaussian blur (spatial smoothing) before segmentation and Haar
FILTER_MORPH = True          # morphological opening/closing on the motion mask
FILTER_AREA = True           # reject blobs smaller than MIN_BLOB_AREA
FILTER_TRAJECTORY = True     # least-squares smoothing of the centroid track (False = last - first point)
SAMPLE_FRAME = 60            # processed-frame index whose motion mask is saved for figures

# Every setting combination gets its own output files, so runs never overwrite each other.
TAG = (f"blur{int(FILTER_BLUR)}_morph{int(FILTER_MORPH)}_area{int(FILTER_AREA)}"
       f"_traj{int(FILTER_TRAJECTORY)}_votes{VOTES_TO_FLAG}")
OUTPUT_PATH = f'{BASE}/annotated_{TAG}.mp4'
SNAP_DIR = f'{BASE}/violations_{TAG}'

CASCADES = {
    'car': 'https://raw.githubusercontent.com/andrewssobral/vehicle_detection_haarcascades/master/cars.xml',
    'two_wheeler': 'https://raw.githubusercontent.com/AdityaPai2398/Vehicle-And-Pedestrian-Detection-Using-Haar-Cascades/master/Main%20Project/Main%20Project/Bike%20Detection/two_wheeler.xml',
    'bus': 'https://raw.githubusercontent.com/AdityaPai2398/Vehicle-And-Pedestrian-Detection-Using-Haar-Cascades/master/Main%20Project/Main%20Project/Bus%20Detection/Bus_front.xml',
}


# %% CELL 2 - Pipeline
def load_cascades():
    out = {}
    if not hasattr(cv2, 'CascadeClassifier'):
        print(f'WARNING: OpenCV {cv2.__version__} has no CascadeClassifier (OpenCV 5 moved it to contrib).\n'
              '  Fix: run  !pip install "opencv-python-headless<5"  then Runtime > Restart session.\n'
              '  Continuing in MOTION-ONLY mode (no Haar), so do not use this run for the Haar results.')
        return out
    for name, url in CASCADES.items():
        f = f'{name}.xml'
        try:
            if not os.path.exists(f):
                urllib.request.urlretrieve(url, f)
            c = cv2.CascadeClassifier(f)
            if not c.empty():
                out[name] = c
        except Exception as e:
            print(f'Could not load {name}: {e}')
    print('Cascades loaded:', list(out))
    return out


def detect(gray, fg, cascades):
    """Moving blobs (background subtraction) classified by Haar cascades on small crops."""
    dets = []
    contours, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in contours:
        if cv2.contourArea(c) < (MIN_BLOB_AREA if FILTER_AREA else 20):   # 20 = ignore 1-pixel specks only
            continue
        x, y, w, h = cv2.boundingRect(c)
        roi = gray[max(0, y - 8):y + h + 8, max(0, x - 8):x + w + 8]   # Haar only runs here
        label = next((n for n, cas in cascades.items()
                      if len(cas.detectMultiScale(roi, 1.1, 3, minSize=(16, 16)))), None)
        if label or USE_BLOB_FALLBACK:
            dets.append((x, y, w, h, label or 'vehicle'))
    return dets


class Track:
    def __init__(self, tid, box, label):
        self.id, self.label = tid, label
        self.hist = deque(maxlen=HIST_LEN)
        self.missed = self.votes = self.age = 0
        self.path = 0.0
        self.box = None
        self.flagged = self.saved = False
        self.add(box)

    def add(self, box):
        x, y, w, h = box
        if self.box is not None and self.hist:
            ratio = (w * h) / max(1, self.box[2] * self.box[3])
            if ratio > MAX_AREA_JUMP or ratio < 1 / MAX_AREA_JUMP:
                self.hist.clear()                    # blob merged/split: old centroids are misleading
                self.votes = 0
        self.box = box
        c = (x + w / 2, y + h / 2)
        if self.hist:
            self.path += np.hypot(c[0] - self.hist[-1][0], c[1] - self.hist[-1][1])
        self.age += 1
        self.hist.append(c)

    def predict(self):
        """Expected centroid now: last position + recent velocity * frames since last seen."""
        k = min(4, len(self.hist))
        if k < 2:
            return self.hist[-1]
        vx = (self.hist[-1][0] - self.hist[-k][0]) / (k - 1)
        vy = (self.hist[-1][1] - self.hist[-k][1]) / (k - 1)
        steps = 1 + self.missed
        return (self.hist[-1][0] + vx * steps, self.hist[-1][1] + vy * steps)

    def judge(self, vec):
        """'wrong', 'ok' or None, from the least-squares motion direction vs. road vector."""
        if len(self.hist) < MIN_HIST:
            return None
        pts = np.array(self.hist, dtype=float)
        if FILTER_TRAJECTORY:
            disp = np.polyfit(np.arange(len(pts)), pts, 1)[0] * (len(pts) - 1)   # smoothed (dx, dy)
        else:
            disp = pts[-1] - pts[0]                                              # raw endpoints
        dist = np.hypot(*disp)
        if dist < MIN_TRAVEL_RATIO * self.box[3]:
            return None
        cos = disp @ np.array(vec, float) / (dist * np.hypot(*vec))
        return 'wrong' if cos < -COS_THR else 'ok' if cos > COS_THR else None


def update_tracks(tracks, dets, next_id):
    """Greedy nearest-centroid matching of detections to existing tracks."""
    items = list(tracks.values())
    preds = [t.predict() for t in items]
    gap = lambda p, d: np.hypot(p[0] - (d[0] + d[2] / 2), p[1] - (d[1] + d[3] / 2))
    pairs = sorted((gap(preds[i], d), i, j) for i, t in enumerate(items) for j, d in enumerate(dets)
                   if gap(preds[i], d) < max(30, 1.5 * d[3]))
    used_t, used_d = set(), set()
    for _, i, j in pairs:
        if i in used_t or j in used_d:
            continue
        if items[i].missed:                          # re-found after a dropout: it may be a different vehicle
            items[i].hist.clear(); items[i].votes = 0
        items[i].add(dets[j][:4]); items[i].label = dets[j][4]; items[i].missed = 0
        used_t.add(i); used_d.add(j)
    for i, t in enumerate(items):
        if i not in used_t:
            t.missed += 1
    for j, d in enumerate(dets):
        if j not in used_d:
            tracks[next_id] = Track(next_id, d[:4], d[4]); next_id += 1
    for k in [k for k, t in tracks.items() if t.missed > MAX_MISSED]:
        del tracks[k]
    return next_id


def vector_at(pt):
    """Allowed vector at point pt (None if outside every lane)."""
    if not LANES:
        return ALLOWED_VECTOR
    for lane in LANES:
        if cv2.pointPolygonTest(np.array(lane['polygon'], np.int32), pt, False) >= 0:
            return lane['vector']
    return None


# %% CELL 3 - Run
def run():
    cascades = load_cascades()
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise FileNotFoundError(f'Cannot open {VIDEO_PATH}')
    os.makedirs(SNAP_DIR, exist_ok=True)
    for old in glob.glob(f'{SNAP_DIR}/violation_*.jpg'):   # clear snapshots from earlier runs
        os.remove(old)
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 25
    skip = max(1, round(src_fps / TARGET_FPS))      # 30 fps -> every 3rd frame, 120 fps -> every 12th
    fps = src_fps / skip
    print(f'Source {src_fps:.0f} fps -> analysing every {skip}th frame ({fps:.1f} fps)')
    out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))

    mog = cv2.createBackgroundSubtractorMOG2(history=150, varThreshold=32, detectShadows=True)
    clahe = cv2.createCLAHE(2.0, (8, 8))
    k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

    tracks, next_id, violators, seen = {}, 0, set(), set()
    n = processed = analysed = blobs = skipped = 0
    t0 = time.time()
    while cap.grab():                      # grab() skips decoding work for dropped frames
        n += 1
        if n % skip:
            continue
        ok, frame = cap.retrieve()
        if not ok:
            break
        frame = cv2.resize(frame, (W, H), interpolation=cv2.INTER_AREA)
        src = cv2.GaussianBlur(frame, (5, 5), 0) if FILTER_BLUR else frame   # blur BEFORE segmentation
        gray = clahe.apply(cv2.cvtColor(src, cv2.COLOR_BGR2GRAY))

        fg = mog.apply(src)
        processed += 1
        if processed <= WARMUP:
            out.write(frame)
            continue
        fg = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)[1]          # drop shadows (127)
        if FILTER_MORPH:
            fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, k_open)           # remove speckle
            fg = cv2.dilate(cv2.morphologyEx(fg, cv2.MORPH_CLOSE, k_close), k_open)  # fill holes
        for x1, y1, x2, y2 in IGNORE_RECTS:                             # popups, timestamp, logo
            fg[y1:y2, x1:x2] = 0
        if processed == SAMPLE_FRAME:                                   # same frame in every run
            cv2.imwrite(f'{BASE}/mask_{TAG}.jpg', fg)
        if (fg > 0).mean() > MAX_FG_FRACTION:                           # global lighting change
            skipped += 1
            out.write(frame)
            continue

        dets = detect(gray, fg, cascades)
        analysed += 1
        blobs += len(dets)
        next_id = update_tracks(tracks, dets, next_id)

        for t in tracks.values():
            if t.missed:
                continue
            if t.age >= MIN_AGE_TO_COUNT and t.path >= MIN_PATH_TO_COUNT:
                seen.add(t.id)                       # count only confirmed, actually-moving tracks
            vec = vector_at(t.hist[-1])
            bx, by, bw, bh = t.box
            at_edge = bx <= EDGE_MARGIN or by <= EDGE_MARGIN or bx + bw >= W - EDGE_MARGIN or by + bh >= H - EDGE_MARGIN
            verdict = t.judge(vec) if vec and not at_edge else None   # cut-off boxes give fake motion
            t.votes = t.votes + 1 if verdict == 'wrong' else 0 if verdict == 'ok' else t.votes
            if t.votes >= VOTES_TO_FLAG:
                t.flagged = True
                violators.add(t.id)
                seen.add(t.id)

            x, y, w, h = t.box
            color = (0, 0, 255) if t.flagged else (0, 255, 0)
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.putText(frame, f"#{t.id} {'WRONG WAY' if t.flagged else 'OK'} [{t.label}]",
                        (x, max(12, y - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            cv2.polylines(frame, [np.array(t.hist, np.int32).reshape(-1, 1, 2)], False, color, 1)
            if t.flagged and not t.saved:
                cv2.imwrite(f'{SNAP_DIR}/violation_id{t.id}_f{n}.jpg', frame)
                t.saved = True

        for lane in LANES:
            cv2.polylines(frame, [np.array(lane['polygon'], np.int32)], True, (255, 200, 0), 1)
        cv2.putText(frame, f'Vehicles: {len(seen)}  Wrong-way: {len(violators)}', (8, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 200, 255), 2)
        out.write(frame)

    cap.release(); out.release()
    secs = time.time() - t0
    print(f'Settings: {TAG}')
    print(f'Done: {processed} frames in {secs:.1f}s ({processed / secs:.1f} fps) | '
          f'vehicles: {len(seen)} (raw tracks created: {next_id}) | wrong-way: {len(violators)} | '
          f'avg detections/frame: {blobs / max(1, analysed):.2f} | flicker frames skipped: {skipped}')
    print(f'Video: {OUTPUT_PATH}')


# %% CELL 4 (optional) - save a gridded reference frame to read lane coordinates
def save_reference_frame(at=60):
    cap = cv2.VideoCapture(VIDEO_PATH); cap.set(cv2.CAP_PROP_POS_FRAMES, at)
    f = cv2.resize(cap.read()[1], (W, H)); cap.release()
    for x in range(0, W, 50):
        cv2.line(f, (x, 0), (x, H), (255, 255, 255), 1); cv2.putText(f, str(x), (x + 2, 10), 0, 0.3, (0, 255, 255), 1)
    for y in range(0, H, 50):
        cv2.line(f, (0, y), (W, y), (255, 255, 255), 1); cv2.putText(f, str(y), (2, y - 2), 0, 0.3, (0, 255, 255), 1)
    cv2.imwrite(f'{BASE}/reference_frame.jpg', f)
# save_reference_frame()


run()


# %% CELL 5 - Show the wrong-way snapshots inside the notebook
def show_violations():
    files = sorted(glob.glob(f'{SNAP_DIR}/violation_*.jpg'))
    if not files:
        print('No wrong-way vehicles were flagged.')
    try:
        from google.colab.patches import cv2_imshow
        show = cv2_imshow
    except ImportError:
        import matplotlib.pyplot as plt
        def show(img):
            plt.figure(figsize=(8, 4.5)); plt.axis('off')
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.show()
    for f in files:
        print(os.path.basename(f))
        show(cv2.imread(f))

show_violations()


Cascades loaded: ['car', 'two_wheeler', 'bus']
Source 30 fps -> analysing every 3th frame (10.0 fps)
Settings: blur1_morph1_area1_traj1_votes3
Done: 113 frames in 12.6s (9.0 fps) | vehicles: 12 (raw tracks created: 34) | wrong-way: 0 | avg detections/frame: 3.95 | flicker frames skipped: 3
Video: /content/drive/MyDrive/semester 3/Digital Image Processing/final project/annotated_blur1_morph1_area1_traj1_votes3.mp4
No wrong-way vehicles were flagged.


In [3]:
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless
!pip install "opencv-python-headless<5"

Found existing installation: opencv-python-headless 4.14.0.94
Uninstalling opencv-python-headless-4.14.0.94:
  Successfully uninstalled opencv-python-headless-4.14.0.94
  Using cached opencv_python_headless-4.14.0.94-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
Using cached opencv_python_headless-4.14.0.94-cp37-abi3-manylinux_2_28_x86_64.whl (62.0 MB)


In [1]:
import cv2
print(cv2.__version__, hasattr(cv2, 'CascadeClassifier'))   # expect 4.x and True

4.14.0 True
